In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier



In [2]:
df = pd.read_csv('train.csv')

df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

# pipeline

In [ ]:
df.drop(columns=["PassengerId","Name","Ticket","Cabin"],inplace=True)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=["Survived"]),
                                                df["Survived"],test_size=0.2,
                                                random_state=42)

In [ ]:
X_train.head()

In [ ]:
y_train.sample(5)

In [ ]:
# imputation tColumnTransformer
trf1 = ColumnTransformer([('impute_age',SimpleImputer(),[2]),
                          ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])],
                         remainder='passthrough')

In [ ]:
trf1

In [ ]:
# One hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),[1,6])
    
],remainder='passthrough')

In [ ]:
trf2

In [ ]:
# Scalling
trf3 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0, 10))
])

In [ ]:
trf3

In [ ]:
# Feature selection 
trf4= SelectKBest(score_func=chi2,k=8)

In [ ]:
trf4

In [ ]:
trf5 = DecisionTreeClassifier()

In [ ]:
trf5

# Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
])

In [ ]:
pipe

# Make Pipeline

In [ ]:
# Alternate SSyntax
pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [ ]:
pipe

In [ ]:
pipe.fit(X_train,y_train)

# Explore the pipeline

In [ ]:
#  code here 
pipe.named_steps

In [ ]:
pipe.named_steps['columntransformer-1']

In [ ]:
pipe.named_steps['columntransformer-1'].transformers_[0][1].statistics_

In [ ]:
pipe.named_steps['columntransformer-1'].transformers_[1][1].statistics_

In [ ]:
# Display pPipeline
from sklearn import set_config
set_config(display='diagram')

In [ ]:
# predict 
y_pred = pipe.predict(X_test)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

# Cross Validation using Pipeline

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,X_train,y_train,cv=5,scoring='accuracy').mean()

# GridSearch Usinng Pipeline

In [ ]:
params={
    'decisiontreeclassifier__max_depth':[1,2,3,4,5,None]
}

In [ ]:
params

In [ ]:
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(pipe,params,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)

In [ ]:
grid.best_score_


In [ ]:
grid.best_params_

In [ ]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))